# Схильності по сегментах → аркуш «Схильності» у звіті

Самостійний ноутбук: не потребує змін у `utils.py` чи `report.ipynb`.
Бере `output/customer_segments.csv.gz`, підтягує залишки, доходи, скоринги і Finalta,
рахує таблицю по сегментах і **додає аркуш «Схильності» в готовий `output/segmentation_report.xlsx`**
(якщо аркуш уже є — перезаписує його).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

## 0. Налаштування

In [ ]:
DATA_PATH = Path(r'C:\repo\data-science\Nazar Hnatyshyn\RETAIL\(DS-1052) Vehicle market analysis\df_for_clustering.xlsx')
OUTPUT_DIR = Path("output")
REPORT_PATH = OUTPUT_DIR / "segmentation_report.xlsx"
SHEET_TITLE = "Схильності"

MONTH_DATA = "2026_10"      # залишки, доходи, Finalta
MONTH_SCORING = "2026_11"   # скоринги депозитів і ОВДП

BASE = Path(r"M:\Controlling\Data_Science_Projects")
LIABILITIES_PATH = BASE / "Common_Data" / "Processed" / f"dataset_{MONTH_DATA}.parquet"
INCOME_PATH = BASE / "Common_Data" / "Raw" / f"info_dataset_{MONTH_DATA}.parquet"
DEPOSIT_PATH = BASE / "Scorings" / MONTH_SCORING / f"deposits_propensity_{MONTH_SCORING}.parquet"
OVDP_PATH = BASE / "Scorings" / MONTH_SCORING / f"dgb_{MONTH_SCORING}.parquet"
FINALTA_PATH = BASE / "Activity" / f"finalta_{MONTH_DATA}.csv"

INCOME_TYPES = {
    "SAL": "Зарплата", "SOC": "Соціальні виплати", "INT": "Відсотки", "BNS": "Бонуси",
    "RNT": "Оренда", "DIV": "Дивіденди", "ROY": "Роялті",
}

for path in [DATA_PATH, REPORT_PATH, LIABILITIES_PATH, INCOME_PATH, DEPOSIT_PATH, OVDP_PATH, FINALTA_PATH]:
    print("OK     " if path.exists() else "НЕМАЄ  ", path)

## 1. Сегменти і вартість ТЗ

In [ ]:
segments = pd.read_csv(OUTPUT_DIR / "customer_segments.csv.gz", usecols=["user_id", "FINAL_SEGMENT"])
segments["user_id"] = segments["user_id"].astype("int64")

raw = pd.read_excel(DATA_PATH, engine="openpyxl", usecols=["user_id", "MARKET_VALUE_USD"])
all_value = (raw.assign(MARKET_VALUE_USD=pd.to_numeric(raw["MARKET_VALUE_USD"], errors="coerce"))
                .groupby("user_id")["MARKET_VALUE_USD"].sum().rename("ALL_VALUE"))
segments = segments.merge(all_value, left_on="user_id", right_index=True, how="left")
print(f"Клієнтів: {len(segments):,} | сегментів: {segments['FINAL_SEGMENT'].nunique()}")

## 2. Зовнішні дані

In [ ]:
def load_keyed(df, key, how):
    """Приводить ключ до user_id (int64) і прибирає дублікати."""
    df = df.rename(columns={key: "user_id"})
    df["user_id"] = df["user_id"].astype("int64")
    dup = int(df["user_id"].duplicated().sum())
    if dup:
        print(f"  дублікатів ключа: {dup:,} → {how}")
        df = df.groupby("user_id", as_index=False).agg(how)
    return df


print("Залишки")
liab = load_keyed(pd.read_parquet(LIABILITIES_PATH, columns=["CONTRAGENTID", "LIABILITIES"]), "CONTRAGENTID", "sum")
liab["LIABILITIES"] = liab["LIABILITIES"].astype("float64")

print("Доходи")
income_cols = {f"{code}_{MONTH_DATA}": code for code in INCOME_TYPES}
inc = pd.read_parquet(INCOME_PATH, columns=["CONTRAGENTID"] + list(income_cols)).rename(columns=income_cols)
inc = load_keyed(inc, "CONTRAGENTID", "sum")
inc["TOTAL_INCOME"] = inc[list(INCOME_TYPES)].fillna(0).sum(axis=1)

print("Депозити")
dep = pd.read_parquet(DEPOSIT_PATH, columns=["CONTRAGENTID", "PROB_TO_BUY"]).rename(columns={"PROB_TO_BUY": "PROB_TO_DEPOSIT"})
dep = load_keyed(dep, "CONTRAGENTID", "first")

print("ОВДП")
ovdp = pd.read_parquet(OVDP_PATH, columns=["CONTRAGENTID", "PROB_TO_BUY"]).rename(columns={"PROB_TO_BUY": "PROB_TO_OVDP"})
ovdp = load_keyed(ovdp, "CONTRAGENTID", "first")

print("Finalta")
active_ids = set(pd.read_csv(FINALTA_PATH, usecols=["CLIENT_ID"])["CLIENT_ID"].astype("int64"))

for name, df in [("liabilities", liab), ("income", inc), ("deposit", dep), ("ovdp", ovdp)]:
    print(f"{name:12s} {df.shape}")
    display(df.head(3))
print("finalta      клієнтів:", f"{len(active_ids):,}")

In [ ]:
fin = (segments.merge(liab, on="user_id", how="left")
               .merge(inc, on="user_id", how="left")
               .merge(dep, on="user_id", how="left")
               .merge(ovdp, on="user_id", how="left"))
fin["IS_ACTIVE"] = fin["user_id"].isin(active_ids)
assert len(fin) == len(segments)

coverage = pd.DataFrame({
    "Джерело": ["Залишки", "Доходи", "Скоринг депозитів", "Скоринг ОВДП", "Finalta (активні)"],
    "Знайдено клієнтів": [fin["LIABILITIES"].notna().sum(), fin["TOTAL_INCOME"].notna().sum(),
                          fin["PROB_TO_DEPOSIT"].notna().sum(), fin["PROB_TO_OVDP"].notna().sum(), fin["IS_ACTIVE"].sum()],
})
coverage["% наших клієнтів"] = (coverage["Знайдено клієнтів"] / len(fin) * 100).round(2)
print("Покриття джерел (скільки наших клієнтів знайдено у файлі):")
display(coverage)

## 3. Таблиця по сегментах

In [ ]:
SEGMENT_DISPLAY = {
    "LUXURY": "Luxury Vehicle Owners", "MODERN_PREMIUM": "Modern Premium Owners",
    "ESTABLISHED_PREMIUM": "Established Premium Owners", "ESTABLISHED_PREMIUM_SUV": "Established Premium SUV Owners",
    "MULTI_CAR_3PLUS": "Multi-Car Owners (3+)", "MULTI_VEHICLE_HOUSEHOLDS": "Multi-Vehicle Households",
    "MAINSTREAM_SUV": "Mainstream SUV Owners", "EV_ADOPTERS": "Electric Vehicle Adopters",
    "CORE_MASS_MARKET_LOWER_MID": "Core Mass Market Lower-Mid", "CORE_MASS_MARKET_BUDGET": "Core Mass Market Budget",
    "BUDGET_LEGACY": "Budget Legacy Owners", "BUSINESS_VEHICLE_OWNERS": "Business Vehicle Owners",
    "COMMERCIAL_VEHICLE_OWNERS": "Commercial Vehicle Owners", "MOTORCYCLE_OWNERS": "Motorcycle Owners",
    "TRAILER_ONLY_OWNERS": "Trailer-Only Owners", "OTHER_NON_PASSENGER": "Other (без легкового авто)",
}
SEGMENT_GROUPS = {
    "Заможні": ["LUXURY", "MODERN_PREMIUM", "ESTABLISHED_PREMIUM_SUV", "MULTI_CAR_3PLUS"],
    "Established Premium": ["ESTABLISHED_PREMIUM"],
    "Масові": ["CORE_MASS_MARKET_LOWER_MID", "MAINSTREAM_SUV", "MULTI_VEHICLE_HOUSEHOLDS", "EV_ADOPTERS"],
    "Бюджетні": ["CORE_MASS_MARKET_BUDGET", "BUDGET_LEGACY"],
    "Бізнес": ["BUSINESS_VEHICLE_OWNERS", "COMMERCIAL_VEHICLE_OWNERS"],
    "Без легкового авто": ["MOTORCYCLE_OWNERS", "TRAILER_ONLY_OWNERS", "OTHER_NON_PASSENGER"],
}
GROUP_COLORS = {"Заможні": "EDE7F6", "Established Premium": "F8E5F0", "Масові": "E3EEFA",
                "Бюджетні": "FFF6DA", "Бізнес": "E6F2E6", "Без легкового авто": "EDEDED", "Інші": "FFFFFF"}
group_of = {seg: grp for grp, segs in SEGMENT_GROUPS.items() for seg in segs}

g = fin.groupby("FINAL_SEGMENT")
n = g.size()


def positive(col):
    """Кількість і % клієнтів зі значенням > 0, медіана серед них."""
    pos = fin[col].gt(0)
    count = pos.groupby(fin["FINAL_SEGMENT"]).sum()
    median = fin.loc[pos].groupby("FINAL_SEGMENT")[col].median()
    return count, count / n, median


cols = {
    ("Сегмент", "Група"): pd.Series({s: group_of.get(s, "Інші") for s in n.index}),
    ("Сегмент", "Сегмент"): pd.Series({s: SEGMENT_DISPLAY.get(s, s) for s in n.index}),
    ("Сегмент", "Клієнтів"): n,
    ("Сегмент", "Медіана вартості всіх ТЗ, USD"): g["ALL_VALUE"].median(),
    ("Активні (Finalta)", "Кількість"): g["IS_ACTIVE"].sum(),
    ("Активні (Finalta)", "%"): g["IS_ACTIVE"].mean(),
}
for col, block in [("LIABILITIES", "Залишки, UAH"), ("TOTAL_INCOME", "Total income, UAH")]:
    count, pct, median = positive(col)
    cols[(block, "Клієнтів > 0")] = count
    cols[(block, "% > 0")] = pct
    cols[(block, "Медіана (> 0)")] = median
for code_, name in INCOME_TYPES.items():
    _, pct, median = positive(code_)
    cols[(f"{name}, UAH", "% > 0")] = pct
    cols[(f"{name}, UAH", "Медіана (> 0)")] = median
for col, name in [("PROB_TO_DEPOSIT", "Схильність до депозиту"), ("PROB_TO_OVDP", "Схильність до ОВДП")]:
    cols[(name, "Медіана")] = g[col].median()
    cols[(name, "% зі скором")] = g[col].apply(lambda s: s.notna().mean())

table = pd.DataFrame(cols)
table.columns = pd.MultiIndex.from_tuples(table.columns)
table = table.sort_values(("Сегмент", "Медіана вартості всіх ТЗ, USD"), ascending=False)
display(table)

## 4. Додати аркуш у звіт

In [ ]:
thin = Side(style="thin", color="B4C6E7")
BORDER = Border(left=thin, right=thin, top=thin, bottom=thin)
CENTER = Alignment(horizontal="center", vertical="center", wrap_text=True)
LEFT = Alignment(horizontal="left", vertical="center", wrap_text=True)
DARK = PatternFill("solid", fgColor="1F3864")
BLUE = PatternFill("solid", fgColor="5B8FD4")
WHITE_BOLD = Font(bold=True, color="FFFFFF")


def number_format(block, metric):
    if metric.startswith("%"):
        return "0.0%"
    if block.startswith("Схильність"):
        return "0.000"
    return "#,##0"


wb = load_workbook(REPORT_PATH)
if SHEET_TITLE in wb.sheetnames:
    del wb[SHEET_TITLE]
ws = wb.create_sheet(SHEET_TITLE)
ws.sheet_view.showGridLines = False

n_cols = table.shape[1]
widths = [2, 18, 32] + [13] * (n_cols - 2)
for i, w in enumerate(widths, start=1):
    ws.column_dimensions[get_column_letter(i)].width = w

# Заголовок і примітка
ws.merge_cells(start_row=1, start_column=2, end_row=1, end_column=1 + n_cols)
ws.cell(row=1, column=2, value="СХИЛЬНОСТІ ТА ФІНАНСОВИЙ ПРОФІЛЬ СЕГМЕНТІВ")
for c in range(2, 2 + n_cols):
    ws.cell(row=1, column=c).fill = DARK
ws.cell(row=1, column=2).font = Font(bold=True, color="FFFFFF", size=16)
ws.cell(row=1, column=2).alignment = CENTER
ws.row_dimensions[1].height = 30
ws.merge_cells(start_row=2, start_column=2, end_row=2, end_column=1 + n_cols)
ws.cell(row=2, column=2, value=(
    "Сегменти відсортовано за медіаною вартості всіх ТЗ. Медіани залишків і доходів — серед клієнтів зі значенням > 0; "
    "медіани схильностей — серед клієнтів зі скором. Total income = зарплата + соцвиплати + відсотки + бонуси + оренда + дивіденди + роялті."))
ws.cell(row=2, column=2).alignment = LEFT
ws.row_dimensions[2].height = 30

# Двоярусна шапка: блоки (рядок 4) і показники (рядок 5)
HEAD1, HEAD2, FIRST = 4, 5, 6
blocks = [c[0] for c in table.columns]
start = 0
while start < n_cols:
    end = start
    while end + 1 < n_cols and blocks[end + 1] == blocks[start]:
        end += 1
    if end > start:
        ws.merge_cells(start_row=HEAD1, start_column=2 + start, end_row=HEAD1, end_column=2 + end)
    ws.cell(row=HEAD1, column=2 + start, value=blocks[start])
    for j in range(start, end + 1):
        cell = ws.cell(row=HEAD1, column=2 + j)
        cell.fill, cell.font, cell.alignment, cell.border = DARK, WHITE_BOLD, CENTER, BORDER
    start = end + 1
for j, (block, metric) in enumerate(table.columns):
    cell = ws.cell(row=HEAD2, column=2 + j, value=metric)
    cell.fill, cell.font, cell.alignment, cell.border = BLUE, WHITE_BOLD, CENTER, BORDER
ws.row_dimensions[HEAD1].height = 34
ws.row_dimensions[HEAD2].height = 45

# Рядки сегментів
for i, (_, row_values) in enumerate(table.iterrows()):
    r = FIRST + i
    fill = PatternFill("solid", fgColor=GROUP_COLORS.get(row_values[("Сегмент", "Група")], "FFFFFF"))
    for j, (block, metric) in enumerate(table.columns):
        value = row_values[(block, metric)]
        if isinstance(value, (float, np.floating)) and np.isnan(value):
            value = "—"
        elif isinstance(value, np.generic):
            value = value.item()
        cell = ws.cell(row=r, column=2 + j, value=value)
        cell.fill, cell.border = fill, BORDER
        cell.alignment = LEFT if metric in ("Група", "Сегмент") else CENTER
        if value != "—" and metric not in ("Група", "Сегмент"):
            cell.number_format = number_format(block, metric)

ws.freeze_panes = ws.cell(row=FIRST, column=4)
wb.save(REPORT_PATH)
print(f"Аркуш «{SHEET_TITLE}» додано: {REPORT_PATH.resolve()}")
print("Аркуші у файлі:", wb.sheetnames)